# 因子测试 · 02：融合因子与 relative 因子折线图（标注国家队出手公告日）

1. 读取 **factor_build** 输出：04_fusion_timeseries、04_fusion_constituents、02_relative_factors_timeseries
2. 按顺序生成文件夹：**01_融合因子**、**02_xxx**、**03_xxx** …（每个为 04 选出的一个 relative 因子）
3. 每个文件夹内画一张该因子时序折线图，图中用 **config.MARK_DATES** 标注国家队出手公告日期
4. 对每个因子、每个国家队出手日，再画一张 **event time** 细图：公告日前后一段时间，一张图内包含大盘-小盘(y)、该因子值、上证50与上证1000（累计涨跌幅）对比

**关于因子连续性**：在 factor_build/02 中，rolling z-score 仅用窗口内有效值（非 0 且非空）算均值和标准差，窗口内至少 2 个有效值即输出 z；相对因子做差后再 **前向填充(ffill)**，故因子一旦开始即为连续序列。

In [1]:
import os
import sys
import shutil
import platform
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

# 中文字体：按系统设置，避免图中中文乱码
system = platform.system()
if system == 'Darwin':
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Heiti SC', 'STHeiti', 'AppleGothic', 'DejaVu Sans']
elif system == 'Windows':
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'KaiTi', 'FangSong', 'Arial Unicode MS', 'DejaVu Sans']
else:
    plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei', 'WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'Droid Sans Fallback', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

_cwd = os.getcwd()
_root = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'factor_test' else _cwd
if _root not in sys.path:
    sys.path.insert(0, _root)
import config

DATE_COL = config.DATE_COL
FB_OUT = os.path.join(_root, 'factor_build', getattr(config, 'FACTOR_BUILD_OUTPUTS', 'outputs'))
FT_OUT = os.path.join(_root, 'factor_test', getattr(config, 'FACTOR_TEST_OUTPUTS', 'outputs'))
MARK_DATES = getattr(config, 'MARK_DATES', ['2023-10-23', '2024-02-06', '2024-09-24', '2025-04-07'])
# 本步输出：在 factor_test/outputs 下建子目录，其下再建 01_融合因子、02_xxx 等
PLOTS_BASE = os.path.join(FT_OUT, '02_factor_plots')

## 1. 读取融合因子与 constituent 列表

In [2]:
path_fusion = os.path.join(FB_OUT, '04_fusion_timeseries.xlsx')
path_const = os.path.join(FB_OUT, '04_fusion_constituents.xlsx')
path_rf = os.path.join(FB_OUT, '02_relative_factors_timeseries.xlsx')
path_y_ts = os.path.join(FB_OUT, '01_y_timeseries.xlsx')
if not os.path.isfile(path_fusion) or not os.path.isfile(path_const) or not os.path.isfile(path_rf):
    raise FileNotFoundError('请先运行 factor_build 的 02 与 04，生成 02_relative_factors_timeseries.xlsx、04_fusion_timeseries.xlsx、04_fusion_constituents.xlsx')

df_fusion = pd.read_excel(path_fusion)
df_fusion[DATE_COL] = pd.to_datetime(df_fusion[DATE_COL]).dt.date
if 'fusion' not in df_fusion.columns:
    raise KeyError('04_fusion_timeseries.xlsx 需包含 fusion 列')

constituents = pd.read_excel(path_const)
if 'sheet' not in constituents.columns or '标的对' not in constituents.columns:
    raise KeyError('04_fusion_constituents.xlsx 需包含 sheet、标的对 列')
const_list = constituents[['sheet', '标的对']].drop_duplicates().reset_index(drop=True)

rf_sheets = pd.read_excel(path_rf, sheet_name=None)
# 规整每个 sheet 的日期列
for name in list(rf_sheets.keys()):
    df = rf_sheets[name]
    if DATE_COL not in df.columns and len(df.columns) >= 1:
        df = df.rename(columns={df.columns[0]: DATE_COL})
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce').dt.date
    rf_sheets[name] = df

# 用于 event 图：大盘-小盘、50/1000 涨跌幅
df_y_ts = None
if os.path.isfile(path_y_ts):
    df_y_ts = pd.read_excel(path_y_ts)
    df_y_ts[DATE_COL] = pd.to_datetime(df_y_ts[DATE_COL]).dt.date
    for c in ['涨跌幅_50', '涨跌幅_1000']:
        if c in df_y_ts.columns:
            df_y_ts[c] = pd.to_numeric(df_y_ts[c], errors='coerce')
    if 'y' not in df_y_ts.columns and '涨跌幅_50' in df_y_ts.columns:
        df_y_ts['y'] = df_y_ts['涨跌幅_50'].astype(float) - df_y_ts['涨跌幅_1000'].astype(float)
    df_y_ts = df_y_ts.sort_values(DATE_COL).reset_index(drop=True)

# ---- 加载沪深300收盘价（用于因子图背景） ----
DATA_DIR = config.get_data_dir()
_csi300_file = next((f for f in os.listdir(DATA_DIR)
                      if all(k in f for k in getattr(config, 'FILE_KEYWORDS_CSI300', ('沪深300', '行情')))
                      and any(f.endswith(ext) for ext in config.EXCEL_EXTENSIONS)
                      and not f.startswith(config.SKIP_FILE_PREFIX)), None)
df_csi50_price = None
if _csi300_file:
    _tmp = pd.read_excel(os.path.join(DATA_DIR, _csi300_file))
    _tmp.columns = _tmp.columns.str.strip()
    _tmp[DATE_COL] = pd.to_datetime(_tmp[DATE_COL], errors='coerce').dt.date
    # 收盘价可能含千分位逗号
    _price_col = '收盘价'
    if _price_col in _tmp.columns:
        _s = _tmp[_price_col]
        if _s.dtype == object or pd.api.types.is_string_dtype(_s):
            _s = _s.astype(str).str.replace(',', '')
        _tmp[_price_col] = pd.to_numeric(_s, errors='coerce')
        df_csi50_price = _tmp[[DATE_COL, _price_col]].dropna().sort_values(DATE_COL).reset_index(drop=True)
        print('沪深300收盘价已加载, 行数:', len(df_csi50_price),
              ', 日期范围:', df_csi50_price[DATE_COL].min(), '~', df_csi50_price[DATE_COL].max())
    else:
        print('沪深300行情中未找到 收盘价 列，跳过背景绘制')
else:
    print('未找到沪深300行情文件，跳过背景绘制')

print('融合因子 行数:', len(df_fusion))
print('04 选出的 relative 因子数:', len(const_list))
print('02 relative 的 sheet 数:', len(rf_sheets))
print('01_y_timeseries 已加载:', df_y_ts is not None and len(df_y_ts) > 0)

沪深300收盘价已加载, 行数: 2431 , 日期范围: 2015-11-13 ~ 2025-11-13
融合因子 行数: 1489
04 选出的 relative 因子数: 4
02 relative 的 sheet 数: 6
01_y_timeseries 已加载: True


## 2. 标注日期转为 date，并定义绘图函数

In [3]:
mark_dates_parsed = []
for s in MARK_DATES:
    try:
        d = datetime.strptime(s.strip(), '%Y-%m-%d').date()
        mark_dates_parsed.append(d)
    except Exception:
        pass
print('标注日期:', mark_dates_parsed)

EVENT_WINDOW = 10  # 公告日前后各 10 个交易日

# 两会区间（从 config 读取）
_lh_raw = getattr(config, 'LIANGHUI_PERIODS', [])
lianghui_periods = [(datetime.strptime(s, '%Y-%m-%d').date(),
                     datetime.strptime(e, '%Y-%m-%d').date()) for s, e in _lh_raw]
print('两会区间数:', len(lianghui_periods))

def plot_series_with_mark_dates(df, date_col, value_col, title, out_path, mark_dates=None,
                                 df_price_bg=None):
    """
    画因子折线图：
    - 背景：沪深300 收盘价（灰色，右轴）
    - 浅蓝色阴影带：历年两会窗口
    - 红色虚线：汇金公告日
    df_price_bg: 沪深300收盘价 DataFrame，含 date_col 和 '收盘价' 两列（可为 None 则不画背景）。
    """
    mark_dates = mark_dates or []
    df = df[[date_col, value_col]].dropna().sort_values(date_col).reset_index(drop=True)
    if len(df) == 0:
        print('无有效数据，跳过:', title)
        return

    fig, ax = plt.subplots(figsize=(14, 5))

    # ---- 背景：上证50 收盘价（右轴，浅灰填充） ----
    ax2 = None
    if df_price_bg is not None and len(df_price_bg) > 0:
        d_min, d_max = df[date_col].min(), df[date_col].max()
        bg = df_price_bg[(df_price_bg[date_col] >= d_min) & (df_price_bg[date_col] <= d_max)].copy()
        if len(bg) > 0:
            ax2 = ax.twinx()
            ax2.fill_between(bg[date_col], bg['收盘价'], alpha=0.10, color='grey')
            ax2.plot(bg[date_col], bg['收盘价'], color='grey', linewidth=0.6, alpha=0.45)
            ax2.set_ylabel('沪深300 收盘价', color='grey', fontsize=9)
            ax2.tick_params(axis='y', labelcolor='grey', labelsize=8)

    # ---- 两会阴影带（浅蓝色） ----
    _lh_labeled = False
    for (s_d, e_d) in lianghui_periods:
        _label = '两会' if not _lh_labeled else ''
        ax.axvspan(s_d, e_d, color='steelblue', alpha=0.13, zorder=0, label=_label)
        _lh_labeled = True

    # ---- 主线：因子值（左轴） ----
    vals = df[value_col].astype(float)
    ax.plot(df[date_col], vals, color='#1f77b4', linewidth=0.9, zorder=3)

    # ---- 汇金公告日竖线 + 标签 ----
    y_top = ax.get_ylim()[1]
    for d in mark_dates:
        ax.axvline(x=d, color='red', linestyle='--', linewidth=0.8, alpha=0.8, zorder=4)
        ax.text(d, y_top, f' {d}', color='red', fontsize=7, rotation=90,
                ha='left', va='top', zorder=5)

    ax.set_title(title)
    ax.set_xlabel(date_col)
    ax.set_ylabel(value_col, color='#1f77b4')
    ax.tick_params(axis='y', labelcolor='#1f77b4')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.xticks(rotation=30)
    ax.legend(loc='upper left', fontsize=8)
    ax.set_zorder(2)
    ax.patch.set_visible(False)
    if ax2 is not None:
        ax2.set_zorder(1)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()
    print('已保存:', out_path)

def plot_event_window(df_y_ts, df_factor, date_col, value_col, mark_date, folder_path, factor_title, window=10):
    """公告日前后 window 日：大盘-小盘、因子、50/1000 累计涨跌幅画在一张图。"""
    if df_y_ts is None or len(df_y_ts) == 0:
        return
    dates_list = df_y_ts[DATE_COL].drop_duplicates().sort_values().tolist()
    if mark_date in dates_list:
        idx = dates_list.index(mark_date)
    else:
        idx = min(len(dates_list)-1, max(0, np.searchsorted(dates_list, mark_date)))
    i0 = max(0, idx - window)
    i1 = min(len(dates_list), idx + window + 1)
    win_dates = dates_list[i0:i1]
    df_win = df_y_ts[df_y_ts[DATE_COL].isin(win_dates)].sort_values(DATE_COL)
    if len(df_win) == 0:
        return
    y_col = 'y' if 'y' in df_win.columns else (df_win.columns[1] if len(df_win.columns) > 1 else None)
    if y_col is None:
        return
    df_f = df_factor[[date_col, value_col]].dropna()
    df_f = df_f[df_f[date_col].isin(win_dates)]
    df_win = df_win.merge(df_f, on=date_col, how='left')
    df_win = df_win.sort_values(date_col).reset_index(drop=True)
    n = len(df_win)
    pos_zero = (df_win[DATE_COL] == mark_date).values
    event_t = np.arange(n) - (np.where(pos_zero)[0][0] if pos_zero.any() else n // 2)
    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax1.plot(event_t, df_win[y_col].astype(float), 'b-', label='大盘-小盘(y)', linewidth=1.2)
    if value_col in df_win.columns:
        ax2 = ax1.twinx()
        ax2.plot(event_t, pd.to_numeric(df_win[value_col], errors='coerce'), color='darkgreen', linewidth=3, label='因子(主)', zorder=5)
        ax2.legend(loc='upper right')
    if '涨跌幅_50' in df_win.columns and '涨跌幅_1000' in df_win.columns:
        cum50 = df_win['涨跌幅_50'].astype(float).cumsum()
        cum1000 = df_win['涨跌幅_1000'].astype(float).cumsum()
        ax1.plot(event_t, cum50, 'c--', label='上证50(累计涨跌幅)', linewidth=0.8)
        ax1.plot(event_t, cum1000, 'm--', label='上证1000(累计涨跌幅)', linewidth=0.8)
    ax1.axvline(x=0, color='red', linestyle='--', alpha=0.8)
    ax1.set_xlabel('Event time (0=公告日)')
    ax1.set_ylabel('y / 累计涨跌幅')
    ax1.legend(loc='upper left')
    ax1.set_title(f'{factor_title} · 公告日 {mark_date}')
    plt.tight_layout()
    out = os.path.join(folder_path, f'event_{mark_date}.png')
    plt.savefig(out, dpi=120)
    plt.close()
    print('已保存:', out)

标注日期: [datetime.date(2023, 10, 23), datetime.date(2024, 2, 6), datetime.date(2024, 9, 24), datetime.date(2025, 4, 7)]
两会区间数: 7


## 3. 建文件夹并依次出图

In [4]:
def safe_folder_name(s):
    """替换不宜做文件夹名的字符。"""
    for c in ['/', '\\', ':', '*', '?', '"', '<', '>', '|']:
        s = s.replace(c, '_')
    return s.strip() or 'unnamed'

# 若输出目录已存在则先清空再重建，避免多出旧文件夹
if os.path.isdir(PLOTS_BASE):
    shutil.rmtree(PLOTS_BASE)
os.makedirs(PLOTS_BASE, exist_ok=True)

# 01_融合因子
folder_01 = os.path.join(PLOTS_BASE, '01_融合因子')
os.makedirs(folder_01, exist_ok=True)
plot_series_with_mark_dates(
    df_fusion, DATE_COL, 'fusion',
    title='融合因子',
    out_path=os.path.join(folder_01, 'fusion.png'),
    mark_dates=mark_dates_parsed,
    df_price_bg=df_csi50_price
)
if df_y_ts is not None:
    for _md in mark_dates_parsed:
        plot_event_window(df_y_ts, df_fusion, DATE_COL, 'fusion', _md, folder_01, '融合因子', window=EVENT_WINDOW)

# 02、03、… 各 relative 因子（与 04_fusion_constituents 顺序一致）
for i, row in const_list.iterrows():
    sheet_name, pair_name = row['sheet'], row['标的对']
    idx = i + 2  # 02, 03, 04, ...
    label = safe_folder_name(f'{sheet_name}_{pair_name}')
    folder_name = f'{idx:02d}_{label}'
    folder_path = os.path.join(PLOTS_BASE, folder_name)
    os.makedirs(folder_path, exist_ok=True)
    # 匹配 sheet：02 写出时 sheet 名可能被截断为 31 字符
    df_sheet = rf_sheets.get(sheet_name)
    if df_sheet is None and len(sheet_name) > 31:
        df_sheet = rf_sheets.get(sheet_name[:31])
    if df_sheet is None:
        for sn in rf_sheets:
            if sn.strip() == sheet_name or sheet_name.startswith(sn):
                df_sheet = rf_sheets[sn]
                break
    if df_sheet is None:
        print('未找到 sheet:', sheet_name, '，跳过')
        continue
    if pair_name not in df_sheet.columns:
        print('未找到列 标的对:', pair_name, '于 sheet', sheet_name, '，跳过')
        continue
    plot_series_with_mark_dates(
        df_sheet, DATE_COL, pair_name,
        title=f'{sheet_name} · {pair_name}',
        out_path=os.path.join(folder_path, 'series.png'),
        mark_dates=mark_dates_parsed,
        df_price_bg=df_csi50_price
    )
    if df_y_ts is not None:
        for _md in mark_dates_parsed:
            plot_event_window(df_y_ts, df_sheet, DATE_COL, pair_name, _md, folder_path, f'{sheet_name}·{pair_name}', window=EVENT_WINDOW)

print('全部完成，输出目录:', PLOTS_BASE)

已保存: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_test/outputs/02_factor_plots/01_融合因子/fusion.png


已保存: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_test/outputs/02_factor_plots/01_融合因子/event_2023-10-23.png
已保存: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_test/outputs/02_factor_plots/01_融合因子/event_2024-02-06.png


已保存: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_test/outputs/02_factor_plots/01_融合因子/event_2024-09-24.png
已保存: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_test/outputs/02_factor_plots/01_融合因子/event_2025-04-07.png


已保存: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_test/outputs/02_factor_plots/02_NAV_iopv_discount_510050.SH-510100.SH/series.png
已保存: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_test/outputs/02_factor_plots/02_NAV_iopv_discount_510050.SH-510100.SH/event_2023-10-23.png
已保存: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_test/outputs/02_factor_plots/02_NAV_iopv_discount_510050.SH-510100.SH/event_2024-02-06.png


已保存: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_test/outputs/02_factor_plots/02_NAV_iopv_discount_510050.SH-510100.SH/event_2024-09-24.png
已保存: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_test/outputs/02_factor_plots/02_NAV_iopv_discount_510050.SH-510100.SH/event_2025-04-07.png


...

已保存: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_test/outputs/02_factor_plots/03_netinflow_159915.SZ-159952.SZ/event_2024-09-24.png
已保存: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_test/outputs/02_factor_plots/03_netinflow_159915.SZ-159952.SZ/event_2025-04-07.png


已保存: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_test/outputs/02_factor_plots/04_turn_588080.SH-588050.SH/series.png
已保存: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_test/outputs/02_factor_plots/04_turn_588080.SH-588050.SH/event_2023-10-23.png
已保存: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_test/outputs/02_factor_plots/04_turn_588080.SH-588050.SH/event_2024-02-06.png


已保存: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_test/outputs/02_factor_plots/04_turn_588080.SH-588050.SH/event_2024-09-24.png
已保存: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_test/outputs/02_factor_plots/04_turn_588080.SH-588050.SH/event_2025-04-07.png


已保存: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_test/outputs/02_factor_plots/05_volume_btin_588080.SH-588050.SH/series.png
已保存: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_test/outputs/02_factor_plots/05_volume_btin_588080.SH-588050.SH/event_2023-10-23.png
已保存: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_test/outputs/02_factor_plots/05_volume_btin_588080.SH-588050.SH/event_2024-02-06.png


已保存: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_test/outputs/02_factor_plots/05_volume_btin_588080.SH-588050.SH/event_2024-09-24.png
已保存: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_test/outputs/02_factor_plots/05_volume_btin_588080.SH-588050.SH/event_2025-04-07.png
全部完成，输出目录: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_test/outputs/02_factor_plots
